# All 3 Models: Single Configuration Test

**Purpose:** Compare LogisticGLM, GRU, and XGBoost on identical data

**Configuration:**
- **Training:** 200 patients
- **Evaluation:** 200 patients (from bootstrap pool)
- **Iterations:** 1 evaluation
- **Models:** LogisticGLM, GRU, XGBoost

**Expected Runtime:** 1-2 minutes (GRU training is slowest)

**Output:** Direct comparison of all three models on the same training/evaluation split

In [ ]:
import sys, logging, traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '.')
logging.basicConfig(level=logging.WARNING)

print("Importing modules...")

from data_loader import (
    load_physionet_files,
    add_hours_until_sepsis,
    split_patients_by_status,
    get_rows_for_patients,
)
from bootstrap import BootstrapResampler
from models import LogisticGLM, GRUModel, XGBoostModel
from training import BootstrapEvaluator

print("✓ All imports successful")

## 1. Load Data

In [ ]:
print("Loading PhysioNet data...")
DATA_DIR = Path('../data/physionet_sepsis')

raw_df = load_physionet_files(DATA_DIR)
print(f"Loaded: {raw_df['patient_id'].nunique():,} patients, {len(raw_df):,} rows")

print("Computing hours_until_sepsis...")
df = add_hours_until_sepsis(raw_df, keep_post_onset=True)
n_septic = df['hours_until_sepsis'].notna().sum()
print(f"Computed: {n_septic:,} septic rows")
print(f"\n✓ Data ready")

## 2. Split Patients (200 train, 200 eval)

In [ ]:
TRAIN_SIZE = 200
EVAL_SIZE = 200
RANDOM_STATE = 42

print(f"\nSplitting patients: {TRAIN_SIZE} train, {EVAL_SIZE} eval")
print(f"(Total {TRAIN_SIZE + EVAL_SIZE} patients, rest discarded)")

# Split: 200 for training, rest for bootstrap pool
train_pids, boot_pids = split_patients_by_status(
    df, n_train_patients=TRAIN_SIZE,
    random_state=RANDOM_STATE, stratify_by_sepsis=True
)
print(f"✓ Training patients: {len(train_pids)}")
print(f"✓ Bootstrap pool: {len(boot_pids)}")

# Get training data
train_df = get_rows_for_patients(df, train_pids)
print(f"\nTraining data:")
print(f"  - {len(train_df):,} rows from {train_pids.nunique()} unique patients")
print(f"  - {(train_df['SepsisLabel'] == 1).sum():,} septic rows ({(train_df['SepsisLabel'] == 1).mean()*100:.1f}%)")

# Create bootstrap resampler for evaluation (will sample 200 patients with replacement)
resampler = BootstrapResampler(
    bootstrap_pool_patient_ids=boot_pids,
    full_df=df,
    n_iterations=1,
    bootstrap_sample_size=EVAL_SIZE,
    random_state=RANDOM_STATE
)

# Generate evaluation sample
_, eval_df = resampler.generate_iteration(0)
print(f"\nEvaluation data:")
print(f"  - {len(eval_df):,} rows from {eval_df['patient_id'].nunique()} unique patients (sampled with replacement)")
print(f"  - {(eval_df['SepsisLabel'] == 1).sum():,} septic rows ({(eval_df['SepsisLabel'] == 1).mean()*100:.1f}%)")
print(f"\n✓ Data split complete")

## 3. Train and Evaluate All Three Models

In [ ]:
print("\n" + "="*80)
print("TRAINING AND EVALUATING MODELS")
print("="*80)

results = []

# Define models to test
models_to_test = [
    ('LogisticGLM', LogisticGLM(C=0.01)),
    ('GRU', GRUModel(hidden_size=64, num_layers=1, dropout=0.2, epochs=10)),
    ('XGBoost', XGBoostModel(n_estimators=200, max_depth=4, learning_rate=0.05)),
]

for model_name, model_instance in models_to_test:
    print(f"\n{'-'*80}")
    print(f"Model: {model_name}")
    print(f"{'-'*80}")
    
    try:
        print(f"\n[1] Creating evaluator and training on {TRAIN_SIZE} patients...")
        evaluator = BootstrapEvaluator(
            model=model_instance,
            train_df=train_df,
            label_column='SepsisLabel',
            patient_id_column='patient_id'
        )
        print(f"✓ Model trained")
        
        print(f"\n[2] Evaluating on {EVAL_SIZE} sampled patients...")
        metrics = evaluator.evaluate_iteration(
            eval_df, 0,
            compute_per_group=True,
            group_column='Gender'
        )
        print(f"✓ Evaluation complete")
        
        # Extract per-group utilities
        utility_f = np.nan
        utility_m = np.nan
        if 'per_group' in metrics:
            if 0 in metrics['per_group']:
                utility_f = metrics['per_group'][0].get('utility', np.nan)
            if 1 in metrics['per_group']:
                utility_m = metrics['per_group'][1].get('utility', np.nan)
        
        row = {
            'model': model_name,
            'train_patients': TRAIN_SIZE,
            'eval_patients': EVAL_SIZE,
            'utility': metrics.get('utility', np.nan),
            'auroc': metrics.get('auroc', np.nan),
            'recall': metrics.get('recall', np.nan),
            'precision': metrics.get('precision', np.nan),
            'f1': metrics.get('f1', np.nan),
            'accuracy': metrics.get('accuracy', np.nan),
            'utility_female': utility_f,
            'utility_male': utility_m,
        }
        results.append(row)
        
        # Print results
        print(f"\n[3] Results:")
        print(f"    Utility:   {metrics.get('utility', np.nan):.4f}")
        print(f"    AUROC:     {metrics.get('auroc', np.nan):.4f}")
        print(f"    Recall:    {metrics.get('recall', np.nan):.4f}")
        print(f"    Precision: {metrics.get('precision', np.nan):.4f}")
        print(f"    F1:        {metrics.get('f1', np.nan):.4f}")
        print(f"    Accuracy:  {metrics.get('accuracy', np.nan):.4f}")
        
        if not np.isnan(utility_f) and not np.isnan(utility_m):
            gap = abs(utility_f - utility_m)
            print(f"    Gender gap: {gap:.4f} (Female: {utility_f:.4f}, Male: {utility_m:.4f})")
        
        print(f"\n✓ {model_name} complete")
        
    except Exception as e:
        print(f"\n✗ {model_name} FAILED: {e}")
        traceback.print_exc()
        results.append({
            'model': model_name,
            'train_patients': TRAIN_SIZE,
            'eval_patients': EVAL_SIZE,
            'utility': np.nan,
            'auroc': np.nan,
            'recall': np.nan,
            'precision': np.nan,
            'f1': np.nan,
            'accuracy': np.nan,
            'utility_female': np.nan,
            'utility_male': np.nan,
        })

print(f"\n\n{'='*80}")
print(f"EVALUATION COMPLETE")
print(f"{'='*80}")

## 4. Comparison Table

In [ ]:
results_df = pd.DataFrame(results)

print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)

print(f"\nFull Results:")
print(results_df[['model', 'utility', 'auroc', 'recall', 'precision', 'f1', 'accuracy']].round(4).to_string(index=False))

print(f"\n\nRanking by Utility Score:")
ranked = results_df.sort_values('utility', ascending=False)[['model', 'utility', 'auroc', 'f1']]
for i, (idx, row) in enumerate(ranked.iterrows(), 1):
    print(f"  {i}. {row['model']:12s} - Utility: {row['utility']:7.4f}, AUROC: {row['auroc']:6.4f}, F1: {row['f1']:6.4f}")

if results_df['utility_female'].notna().any():
    print(f"\n\nGender Fairness (Utility Gap):")
    for idx, row in results_df.iterrows():
        if not np.isnan(row['utility_female']) and not np.isnan(row['utility_male']):
            gap = abs(row['utility_female'] - row['utility_male'])
            print(f"  {row['model']:12s} - Gap: {gap:.4f} (F: {row['utility_female']:.4f}, M: {row['utility_male']:.4f})")

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(f'Model Comparison (Train: {TRAIN_SIZE} patients, Eval: {EVAL_SIZE} patients)', 
             fontsize=14, fontweight='bold')

metrics_to_plot = [
    ('utility', 'Utility'),
    ('auroc', 'AUROC'),
    ('recall', 'Recall'),
    ('precision', 'Precision'),
    ('f1', 'F1 Score'),
    ('accuracy', 'Accuracy'),
]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for ax, (metric, title) in zip(axes.flat, metrics_to_plot):
    values = results_df[metric].values
    models = results_df['model'].values
    
    bars = ax.bar(models, values, color=colors[:len(models)], alpha=0.8, edgecolor='black', linewidth=1.5)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        if not np.isnan(height):
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.4f}',
                   ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_ylabel(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0 if metric in ['utility', 'auroc', 'recall', 'precision', 'f1', 'accuracy'] else None,
                  max(1.0, values.max() * 1.1) if not np.isnan(values).all() else 1.0)

plt.tight_layout()
plt.savefig('model_comparison_single_config.png', dpi=150, bbox_inches='tight')
print("\n✓ Saved: model_comparison_single_config.png")
plt.show()

## 6. Save Results

In [ ]:
results_df.to_csv('model_comparison_single_config.csv', index=False)
print("✓ Saved: model_comparison_single_config.csv")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"\nTrain/Eval Configuration:")
print(f"  Training patients: {TRAIN_SIZE}")
print(f"  Evaluation patients: {EVAL_SIZE} (sampled with replacement)")
print(f"  Models: {', '.join(results_df['model'].values)}")
print(f"\nBest Model by Utility: {results_df.loc[results_df['utility'].idxmax(), 'model']}")
print(f"  Utility score: {results_df['utility'].max():.4f}")
print(f"\nOutputs:")
print(f"  - model_comparison_single_config.csv")
print(f"  - model_comparison_single_config.png")
print(f"\n✓ Done!")